In [1]:
!pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [2]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [3]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [10]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json


END_OF_SHIFT_CLEANING_PROMPT = """
Analyze the quick service restaurant dining room image and return a structured end-of-shift cleaning readiness report.

Return only valid JSON. Do not include markdown, explanations, or extra commentary.

Use this exact JSON structure:

{
  "overall_cleanliness_score": null,
  "dining_room_clean": null,
  "tables_wiped": null,
  "floors_appear_mopped": null,
  "trash_removed": null,
  "condiment_stations_stocked": null,
  "chairs_positioned_correctly": null,
  "lights_in_correct_state": null,
  "cleaning_confidence": null,
  "issues": [],
  "annotated_evidence": []
}

Instructions:

- "overall_cleanliness_score" should be an integer from 1 to 5.
Use 5 when the restaurant appears fully cleaned and ready for tomorrow.
Use 4 when the restaurant is mostly ready with only minor visible issues.
Use 3 when the restaurant is partially cleaned but several issues remain.
Use 2 when the restaurant is mostly not ready.
Use 1 when the restaurant appears very dirty, disorganized, or clearly not ready for tomorrow.

- "dining_room_clean" should be an integer from 1 to 5 based on the overall visible dining room cleanliness.
Use 5 for a clean, organized dining room.
Use 3 for mixed cleanliness with some visible issues.
Use 1 for a dirty or clearly unprepared dining room.

- "tables_wiped" should be an integer from 1 to 5.
Use 5 when tables appear clean, empty, and wiped.
Use 3 when some tables appear clean but others have crumbs, trays, cups, wrappers, or residue.
Use 1 when many tables are visibly dirty or cluttered.

- "floors_appear_mopped" should be an integer from 1 to 5.
Use 5 when floors appear clean, dry, and recently mopped.
Use 3 when floors are mostly clean but have small visible debris or marks.
Use 1 when floors show spills, food, trash, stains, or obvious unmopped areas.

- "trash_removed" should be an integer from 1 to 5.
Use 5 when trash bins appear empty, clean, or not overflowing.
Use 3 when trash status is partially visible or minor trash remains.
Use 1 when trash bins are overflowing, trash bags are left out, or loose trash is visible.

- "condiment_stations_stocked" should be an integer from 1 to 5.
Use 5 when condiment stations appear clean, organized, and stocked.
Use 3 when the station is partially stocked or hard to assess.
Use 1 when the station is visibly empty, messy, dirty, or missing supplies.
Use null if no condiment station is visible.

- "chairs_positioned_correctly" should be an integer from 1 to 5.
Use 5 when chairs are pushed in, aligned, stacked, or positioned correctly for closing.
Use 3 when some chairs are positioned correctly but others are scattered.
Use 1 when many chairs are disorganized, blocking walkways, or left randomly around tables.

- "lights_in_correct_state" should be one of: "correct", "incorrect", "unknown".
Use "correct" when the visible lighting appears appropriate for end-of-shift cleaning or closing readiness.
Use "incorrect" when lights appear obviously wrong for the scene, such as the dining room appearing too dark to inspect or unnecessary lights left on in a closed area.
Use "unknown" if lighting state cannot be assessed from the image.

- "cleaning_confidence" should be one of: "high", "medium", or "low".
Use "high" when the image clearly shows the relevant dining room areas.
Use "medium" when some areas are visible but others are partly blocked or unclear.
Use "low" when the image is blurry, too dark, too cropped, or does not show enough of the dining room.

- "issues" should list short issue labels for all visible problems.
Possible values include: "dirty_tables", "unwiped_tables", "food_on_floor", "trash_visible", "overflowing_trash", "condiment_station_messy", "condiment_station_unstocked", "chairs_misaligned", "chairs_blocking_walkway", "floor_not_mopped", "poor_lighting", "none".

- "annotated_evidence" should be a list of at most 3 visible evidence objects.
Each object should use this structure:
{
  "issue": null,
  "location": null,
  "evidence": null,
  "severity": null
}

For "location", use short approximate visual locations such as "front left tables", "center dining area", "back right condiment station", "near trash bin", "main walkway", or "unknown".
For "evidence", use one short sentence under 15 words.
For "severity", use one of: "minor", "moderate", "severe".
Only include the 3 most important visible issues.

Important rules:
- Only assess what is visually observable.
- Do not guess hidden areas.
- Do not rely on readable signs, captions, timestamps, camera overlays, or UI elements.
- Ignore brand logos and restaurant names.
- If an area is not visible, use null or "unknown" instead of guessing.
- If no issue is visible, use "issues": ["none"] and an empty "annotated_evidence" list.

Return only the JSON object.
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.end-of-shift-cleaning",
        description="Analyze quick service restaurant dining room images and return a structured end-of-shift cleanliness and readiness report, including dining room cleanliness, wiped tables, mopped floors, trash removal, stocked condiment stations, chair positioning, lighting state, confidence, and visible issues",
        worker_release="qwen3-instruct",
        text_prompt=END_OF_SHIFT_CLEANING_PROMPT,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=800,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [11]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a67b07801677798000166294375cf7 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.describe.end-of-shift-cleaning', tag='1.0.3'), AbilityAliasEntry(alias='datasciencealliance-org.describe.end-of-shift-cleaning', tag='latest')]


### Evalulate on a Single Image

In [13]:
from pathlib import Path
import json
from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop


pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.describe.end-of-shift-cleaning:latest"
    )
])


# Replace this with your actual image path.
input_path = Path("/content/sample_end_of_shift_cleaning_image.jpg")


raw_results = []

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    job = endpoint.upload(input_path)

    while result := job.predict():
        raw_results.append(result)


print("=== RAW EYEPOP RESULT ===")
print(json.dumps(raw_results[:1], indent=2))

=== RAW EYEPOP RESULT ===
[
  {
    "seconds": 0,
    "source_height": 768,
    "source_id": "026bfa24-89f1-11f1-966b-622312604b0e",
    "source_width": 1408,
    "system_timestamp": 1785180365973445000,
    "texts": [
      {
        "id": 1,
        "text": "```json\n{\n  \"overall_cleanliness_score\": 1,\n  \"dining_room_clean\": 1,\n  \"tables_wiped\": 1,\n  \"floors_appear_mopped\": 1,\n  \"trash_removed\": 1,\n  \"condiment_stations_stocked\": 3,\n  \"chairs_positioned_correctly\": 1,\n  \"lights_in_correct_state\": \"correct\",\n  \"cleaning_confidence\": \"high\",\n  \"issues\": [\n    \"dirty_tables\",\n    \"unwiped_tables\",\n    \"food_on_floor\",\n    \"trash_visible\",\n    \"condiment_station_messy\",\n    \"chairs_misaligned\",\n    \"floor_not_mopped\"\n  ],\n  \"annotated_evidence\": [\n    {\n      \"issue\": \"dirty_tables\",\n      \"location\": \"front left tables\",\n      \"evidence\": \"Tables are covered with dirty trays, cups, wrappers, and food residue.\",\n